In [ ]:
import pandas as pd
import numpy as np
import iqplot


import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:
fname = '~/git/coalescence-pilot-mgx/workflow/out/midas2_output/species/metadata.tsv'
df_metadata= pd.read_csv(fname, delimiter = '\t')
df_metadata
df_abundance = pd.read_csv('e004Assembly_rel_abundance.csv')
df_abundance 

In [ ]:
def transform_df(df_abundance):
    df_abundance['Lineage'] = df_abundance['species_id'].transform(lambda x: df_metadata.loc[df_metadata['species_id'] == x,'Lineage'].values[0])
    df_abundance['species'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-1])
    df_abundance['genus'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-2])
    df_abundance['family'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-3])
    df_abundance['phyla'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[1])
    return df_abundance

In [ ]:
df_abundance = transform_df(df_abundance)
df_abundance['relative_abundance'] = df_abundance['relative_abundance'].astype(float)

In [ ]:

df_abundance_melted_grouped = df_abundance.groupby(['species']).sum().sort_values(by='relative_abundance', ascending = False)
top_species = df_abundance_melted_grouped.index.drop('s__').values[:20]
top_species

In [ ]:
cmap = bokeh.palettes.Category20[20]
cmap_dict = {}
for i, sp in enumerate(top_species):
    cmap_dict[sp] = cmap[i]
cmap_dict

In [ ]:
def make_stacked_bar_plot(df, cmap_dict):
    top_species = list(cmap_dict.keys())
    
    df_small = df.loc[df['species'].isin(top_species),:]
    bars = hv.Bars(df_small.sort_values(by=['passage','comm']) , kdims=['sample','species'],
               vdims = ['relative_abundance'])

    bars.opts(width=800,height = 800, ).opts(stacked=True, xrotation = 90,cmap = cmap_dict)

    bars.opts(legend_position='left')
    return bars
    

In [ ]:
for p in df_abundance['subject'].unique():
    dfAA = df_abundance.loc[df_abundance['subject'] == p,:]
    p1 = make_stacked_bar_plot(dfAA, cmap_dict)
    bokeh.io.show(hv.render(p1))

In [ ]:
dfAAmGAM.loc[dfAAmGAM['sample'] == 'A8-e003Coalescence-mGAM-p7','parent_subjects'].unique()